# 6장 · HTTP 요청과 JSON 응답 처리

HTTP 요청에서는 경로가 대상을 지정하고 상태 코드가 처리 결과를 알려 줍니다. 성공 응답을 받아도 본문이 예상한 JSON인지 따로 확인해야 합니다.

할 일 1번을 조회하는 예제로 요청 경로, HTTP 상태, JSON 본문, 응답 ID를 검사합니다. 기본 실습은 가상 응답을 사용하며 외부 계정 없이 실행할 수 있습니다.

실행을 확인한 커널은 Python 3.12.15입니다. 예제에 사용한 외부 패키지는 httpx 0.28.1입니다.

Python 3 커널을 선택하고 위에서부터 실행하세요. 과제 셀에 직접 코드를 작성한 뒤 바로 아래에서 풀이를 펼쳐 보세요.


## 경로 매개변수와 요청 ID

GET은 값을 읽어 달라는 요청입니다. /todos/1에서 /todos는 자료 종류, 마지막 1은 요청한 기록 번호입니다. 서버는 상태 코드와 본문을 돌려줍니다.


### 선택 실습 · 공개 API 요청

<p>timeout=5.0은 연결·읽기 등 네트워크 단계의 대기 제한에 사용됩니다. 전체 프로그램이 반드시 5초 안에 끝난다는 보장은 아닙니다. 무제한 대기를 피하려는 설정이며 상세 시간 측정은 별도로 해야 합니다.</p>


### 셀 01 · f-string으로 경로 구성

입력받은 번호를 주소에서 한 번만 사용하도록 만듭니다.


In [1]:
todo_id = 1
path = f"/todos/{todo_id}"
print(path)

/todos/1


/todos/1입니다. 뒤에서 받은 데이터의 id가 1인지 다시 확인할 때 같은 todo_id를 사용합니다.


### HTTP 리소스와 응답

<p>HTTP 요청은 서버에 어떤 자원을 어떻게 다룰지 전달하는 메시지입니다. URL은 주소, GET은 조회 방법입니다. 서버는 상태 코드·헤더·본문을 응답합니다. JSON은 응답 본문에 담길 수 있는 데이터 형식이며 HTTP 자체와 같은 뜻은 아닙니다.</p>

<p>이 장은 인증키가 필요 없는 공개 연습용 할 일 API를 사용합니다. 외부 호출은 조회(GET) 한 건으로 제한합니다. 같은 URL이더라도 서버나 네트워크 상태에 따라 결과가 달라지므로 함수와 실패 처리를 따로 만듭니다. requests도 많이 쓰이지만 이번 실습은 HTTPX 하나로 통일하고 비동기 호출은 다루지 않습니다.</p>

<table><caption>URL·응답 구성</caption><thead><tr><th scope="col">부분</th><th scope="col">예</th><th scope="col">역할</th></tr>
</thead><tbody><tr><td>주소</td><td>https://jsonplaceholder.typicode.com</td><td>연결할 서비스</td></tr>
<tr><td>경로</td><td>/todos/1</td><td>조회할 자원</td></tr>
<tr><td>쿼리</td><td>?userId=1</td><td>조회 조건의 예</td></tr>
<tr><td>상태 코드</td><td>200 / 404</td><td>처리 결과의 종류</td></tr>
<tr><td>헤더</td><td>Content-Type: application/json</td><td>본문 형식 등의 정보</td></tr>
<tr><td>본문</td><td>id·title·completed</td><td>실제 데이터</td></tr>
</tbody></table>

<table><caption>이 장에서 구분할 상태</caption><thead><tr><th scope="col">범위·예</th><th scope="col">뜻</th><th scope="col">대응 예</th></tr>
</thead><tbody><tr><td>2xx / 200</td><td>서버가 성공으로 응답</td><td>본문 형식도 추가 확인</td></tr>
<tr><td>400</td><td>잘못된 요청</td><td>요청 값·문서 확인</td></tr>
<tr><td>401 / 403</td><td>인증 필요 / 접근 거부</td><td>권한·인증 설정 확인</td></tr>
<tr><td>404</td><td>해당 자원 없음</td><td>경로·식별자 확인</td></tr>
<tr><td>429</td><td>요청 제한</td><td>즉시 반복하지 않고 정책 확인</td></tr>
<tr><td>5xx</td><td>서버 측 실패</td><td>기록하고 재시도 정책 확인</td></tr>
</tbody></table>

<h3>조회에서 사용까지</h3>

<p>200만 확인하고 곧바로 화면에 쓰지 않습니다.</p>

<article><small>STEP 01 · 클라이언트</small><h4>GET 경로와 리소스 ID</h4>
<p>타임아웃을 설정하고 요청합니다.</p><pre>요청할 id = 1</pre></article>

<article><small>STEP 02 · HTTPX</small><h4>HTTP 상태 코드 검사</h4>
<p>raise_for_status()로 2xx가 아닌 응답을 구분합니다.</p><pre>200 → 본문 읽기
404 → 상태 검사에서 중단 (셀 05)</pre></article>

<article><small>STEP 03 · JSON</small><h4>JSON 응답 디코딩</h4>
<p>JSON 파싱이 실패할 수도 있습니다.</p><pre>JSON → Python 값
HTML → JSON 변환에서 중단 (셀 04)</pre></article>

<article><small>STEP 04 · 우리 함수</small><h4>필수 키와 필드 타입 검사</h4>
<p>id·title·completed 계약을 확인한 뒤 사용합니다.</p><pre>title = "HTTP 연습"</pre></article>


### 셀 02 · JSON 응답 디코딩

서버가 보냈다고 가정한 JSON 한 건을 Python 값으로 바꿉니다.


In [2]:
import json
body = '{"id":1,"title":"  HTTP 연습  ","completed":false}'
todo = json.loads(body)
print(todo["title"].strip())

HTTP 연습


디코딩한 title 값은 ‘HTTP 연습’입니다.

요청 주소는 1번인데 응답의 id만 2로 바꿔 보세요. JSON을 읽는 줄은 여전히 성공합니다. 잘못된 할 일을 화면에 띄우지 않으려면 해석한 객체의 id와 처음 주소를 만들 때 쓴 todo_id를 대조해야 합니다.


### 과제 03 · 요청 ID와 응답 ID 대조

응답에서 id만 2로 바꿨습니다. 받은 번호가 요청 번호와 같은지 출력하세요.


In [3]:
received = {**todo, "id": 2}
# 받은 번호와 요청한 번호를 비교하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>received = {**todo, &quot;id&quot;: 2}
print(received[&quot;id&quot;] == todo_id)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False이므로 이 제목을 1번 기록으로 표시하면 안 됩니다. 상태 코드가 200인 경우에도 검사할 수 있는 내용입니다.</p>
</details>


### 응답 ID가 원래 요청과 같은지 확인

<p>이 실습은 1번을 요청했지만 received에는 id가 2인 응답을 넣습니다. 응답 본문을 읽을 수 있어도 다른 번호의 자료일 수 있으므로 받은 id와 todo_id를 직접 대조하세요.</p>


## HTTP 상태 오류와 JSONDecodeError

404는 요청에 응답했지만 해당 자원을 찾지 못했다는 상태입니다. 타임아웃이면 읽을 응답 본문조차 없을 수 있습니다. 상태 확인을 먼저 하고 성공했을 때만 JSON을 읽습니다.


### 모의 응답별 분기 테스트

<p>MockTransport는 실제 소켓 통신 대신 handler 함수가 만든 응답을 돌려줍니다. 외부 인터넷이 끊겨도 같은 입력을 반복할 수 있어 실패 처리 학습에 사용합니다. 제공 handler의 경로별 응답을 읽고 바꿔 보는 것이 기본 활동입니다.</p>

<p>모의 타임아웃은 예외를 즉시 발생시킨 것입니다. 실제 서버를 기다려 5초가 지났다는 측정 결과가 아닙니다. 이 검사만으로 DNS·TLS·프록시·실제 서버 접속이 된다고 판단하지 않습니다.</p>


### 셀 04 · JSONDecodeError 처리

상태 코드는 200인데 본문에는 HTML이 담겼다고 가정합니다. 어떤 변환에서 실패하는지 확인하세요.


In [4]:
try:
    json.loads("<html>점검 중</html>")
except json.JSONDecodeError as error:
    print(type(error).__name__)

JSONDecodeError


JSONDecodeError가 나옵니다. URL을 다시 요청하기 전에 이미 받은 본문의 종류와 형식을 점검할 수 있습니다.


### HTTP 상태와 JSON 디코딩 따로 확인하기

<p>서버 상태를 확인하는 raise_for_status()와 본문을 해석하는 json.loads()는 실패 지점이 다릅니다. 점검용 HTML처럼 JSON이 아닌 본문은 JSONDecodeError 경로에서 확인합니다.</p>


### 셀 05 · HTTPX의 상태 코드 예외

httpx가 설치된 수업 커널에서는 MockTransport가 정한 상태로 응답합니다. 네트워크 연결은 없습니다.

client.get(path)를 호출하면 MockTransport가 이 셀의 missing(request)를 대신 부릅니다. Response(404)를 만든 곳은 바로 그 함수입니다. 인터넷을 끊어도 이 404를 관찰할 수 있습니다.


In [5]:
from importlib.util import find_spec
if find_spec("httpx") is not None:
    import httpx
    def missing(request):
        return httpx.Response(404, json={"detail": "할 일을 찾을 수 없습니다."})
    with httpx.Client(transport=httpx.MockTransport(missing), base_url="https://example.invalid") as client:
        response = client.get(path)
        try:
            response.raise_for_status()
        except httpx.HTTPStatusError as error:
            print(error.response.status_code)
else:
    print("HTTPX가 설치된 커널에서 404 응답 실습을 실행하세요.")

404


HTTPX를 사용한 출력은 404입니다. 연결 실패 예외와 달리 error.response에서 서버의 상태 값을 읽습니다. 외부 서버의 가동 여부는 이 모의 요청으로 알 수 없습니다.


### MockTransport가 돌려준 HTTP 응답

<p>예제의 missing 함수는 요청을 받으면 상태 404인 httpx.Response를 만듭니다. 클라이언트의 raise_for_status()에서 HTTPStatusError가 발생하며, except 블록에서 error.response.status_code를 읽으면 404입니다.</p>


## JSON 불리언과 문자열 구분

JSON의 false는 파이썬의 False가 되지만 따옴표가 있는 "false"는 비어 있지 않은 문자열입니다. 자료형을 먼저 검사하면 잘못된 완료 표시를 막을 수 있습니다.


### 과제 06 · bool 타입 검사

잘못 들어온 문자열을 bool()로 바꾸지 말고 완료 값이 실제 bool인지 확인하세요.


In [6]:
completed = "false"
# completed의 자료형을 확인하세요.

<details><summary>정답 코드와 실행 결과 보기</summary>
<pre><code>completed = &quot;false&quot;
print(type(completed) is bool)</code></pre>
<p>실행 결과</p><pre><code>False</code></pre>
<p>False가 나옵니다. 응답 형식 오류로 기록할 입력입니다. bool(completed)를 쓰면 내용과 관계없이 비어 있지 않은 문자열이어서 True가 됩니다.</p>
</details>


### JSON 구조와 필드 검증

<p>외부에서 받은 값은 우리 코드의 타입 힌트대로 온다고 가정하지 않습니다. 이 함수는 JSON 객체(dict), 양의 정수 id, 비어 있지 않은 문자열 title, bool completed를 요구합니다. 필요한 키가 없거나 종류가 다르면 ValueError로 거부합니다.</p>

<p>isinstance(값, 종류)는 해당 종류인지 검사합니다. bool은 Python에서 int의 하위 종류이므로 id에 True가 들어오는 것을 막으려면 type(todo_id) is int처럼 정확한 종류를 확인할 수 있습니다. 여기서는 숫자와 bool을 구분하는 과제 규칙에 사용합니다.</p>

<p>문자열 "false"를 bool(...)로 바꾸면 비어 있지 않아 True가 됩니다. completed를 임의 변환하지 말고 JSON의 true/false가 Python bool로 읽혔는지 확인합니다.</p>


## 돌아보기

주소를 잘못 썼을 때, 기다리는 동안 연결이 끊겼을 때, JSON의 칸이 잘못됐을 때를 차례로 떠올려 보세요. 같은 ‘조회 실패’ 화면에서도 기록할 정보와 다음에 확인할 곳이 달라집니다.

### 조회에서 사용까지

**GET /todos/1**　타임아웃을 설정하고 요청합니다.

```
요청할 id = 1
```

**성공 코드 검사**　raise_for_status()로 2xx가 아닌 응답을 구분합니다.

```
200 → 본문 읽기
404 → 상태 검사에서 중단 (셀 05)
```

**본문을 Python 값으로**　JSON 파싱이 실패할 수도 있습니다.

```
JSON → Python 값
HTML → JSON 변환에서 중단 (셀 04)
```

**필요한 키·타입 검사**　id·title·completed 계약을 확인한 뒤 사용합니다.

```
title = "HTTP 연습"
```


### API 호출 결과와 운영 기록

<table><caption>실패 기록에 남길 내용</caption><thead><tr><th scope="col">항목</th><th scope="col">예</th></tr>
</thead><tbody><tr><td>실행 종류</td><td>모의 / 실제</td></tr>
<tr><td>요청</td><td>GET /todos/1, 공개 연습 데이터</td></tr>
<tr><td>결과</td><td>성공 / 상태 코드 / 형식 오류 / 연결 실패</td></tr>
<tr><td>확인 범위</td><td>JSON 검증까지 / 외부 연결 미확인</td></tr>
<tr><td>다음 행동</td><td>입력 수정·네트워크 확인·수업용 모의 전환</td></tr>
</tbody></table>
